<a href="https://colab.research.google.com/github/elkins/synth-nmr/blob/master/docs/tutorials/relaxation_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Relaxation & Dynamics Analysis

This tutorial demonstrates how to use `synth-nmr` to probe the theoretical internal dynamics of a protein model using the Lipari-Szabo Model-Free formalism and analyze Nuclear Spin Relaxation limits.

## 1. Setup Environment

In [ ]:
!pip install -q synth-nmr biotite matplotlib

## 2. Load Protein Structure

In [ ]:
!wget -q https://files.rcsb.org/download/1D3Z.pdb -O ubiquitin.pdb

import biotite.structure.io as strucio
import matplotlib.pyplot as plt

ensemble = strucio.load_structure("ubiquitin.pdb")
structure = ensemble[0]

## 3. Calculating Relaxation Rates

We need to specify the global tumbling time (`tau_m_ns`) and the strength of the external magnetic field (`field_mhz`). Higher fields increase the contribution of Chemical Shift Anisotropy (CSA) to the $R_1$ and $R_2$ rates.

In [ ]:
from synth_nmr import calculate_relaxation_rates

rates = calculate_relaxation_rates(
    structure,
    field_mhz=600.0, # 600 MHz Spectrometer
    tau_m_ns=4.1     # ~4.1 ns tumbling time for Ubiquitin at 300K
)

print(f"Calculated relaxation parameters for {len(rates)} backbone Amide sites.")

## 4. Visualizing Internal Dynamics (Order Parameters)

The Generalized Order Parameter ($S^2$) ranges from 0 (completely flexible) to 1 (completely rigid). `synth-nmr` predicts $S^2$ based on the protein's secondary structure and structural elements.

In [ ]:
residues = list(rates.keys())
r1_data = [info['R1'] for info in rates.values()]
r2_data = [info['R2'] for info in rates.values()]
noe_data = [info['NOE'] for info in rates.values()]
s2_data = [info['S2'] for info in rates.values()]

plt.figure(figsize=(10, 4))
plt.bar(residues, s2_data, color='teal', alpha=0.7)
plt.ylim(0, 1.05)
plt.xlabel("Residue Number")
plt.ylabel("S² (Order Parameter)")
plt.title("Predicted Backbone Flexibility of Ubiquitin")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

Notice how the termini (start and end of the sequence) are highly flexible (low $S^2$), while the core helices and sheets are very rigid (high $S^2$).